# Module 03: Platform Features

[Workshop home](../README.md) · [Endpoint reference](../docs/inference.md) · [Troubleshooting](../docs/troubleshooting.md)

**25 minutes.** Three capabilities that sit on top of plain chat: image input,
reasoning mode, and typed JSON output.

These are independent of modules 02 and 04. Run them in any order.

| Step | What | Model |
|---|---|---|
| 1 | Send an image and ask about it | `gemma-4-26b-a4b-it` |
| 2 | Find out why a remote URL does not work | `gemma-4-26b-a4b-it` |
| 3 | Turn reasoning on, and watch your token budget | `qwen3-8b` |
| 4 | Get JSON that matches a schema every time | `qwen3-8b` |

## Step 1: Connect

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

root = Path.cwd()
if not (root / ".env").is_file() and (root.parent / ".env").is_file():
    root = root.parent
if not (root / ".env").is_file():
    raise RuntimeError("No .env found. See docs/00_prework.md")

load_dotenv(root / ".env", override=True)
client = OpenAI(
    base_url=os.environ["AKAMAI_BASE_URL"].strip().rstrip("/"),
    api_key=os.environ["AKAMAI_API_KEY"].strip(),
    timeout=180.0,
)

VISION_MODEL = "gemma-4-26b-a4b-it"
CHAT_MODEL = "qwen3-8b"
MODULE_DIR = root / "03_Platform_Features"
print("Ready.")

## Step 2: Send an image

`gemma-4-26b-a4b-it` is the only model here that accepts images. Instead of a plain
string, `content` becomes a list of parts: some text, and an image.

The image goes in as a base64 data URL, which means you read the bytes, encode them,
and build a `data:image/png;base64,...` string.

In [ ]:
import base64
from IPython.display import Image as ShowImage

image_path = MODULE_DIR / "assets" / "shapes.png"
image_bytes = image_path.read_bytes()
data_url = "data:image/png;base64," + base64.b64encode(image_bytes).decode()

print(f"{image_path.name}: {len(image_bytes):,} bytes -> {len(data_url):,} characters encoded")
ShowImage(data=image_bytes)

In [ ]:
reply = client.chat.completions.create(
    model=VISION_MODEL,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "List every shape and its colour. One per line."},
            {"type": "image_url", "image_url": {"url": data_url}},
        ],
    }],
    max_tokens=150,
    temperature=0.0,
)
print(reply.choices[0].message.content)

Base64 inflates the payload by about a third, so a large photo becomes a large
request. Resize before sending if you are doing this at any volume.

Sending an image to a text only model returns a clear error rather than silently
ignoring it. Worth knowing, because the model id is easy to get wrong.

## Step 3: Why a remote URL does not work

The OpenAI API lets you pass an `https://` URL and the service fetches it. That does
not work here, and the failure is worth seeing once so you recognise it.

In [ ]:
try:
    client.chat.completions.create(
        model=VISION_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe this image."},
                {"type": "image_url", "image_url": {
                    "url": "https://upload.wikimedia.org/wikipedia/commons/4/47/PNG_transparency_demonstration_1.png"
                }},
            ],
        }],
        max_tokens=60,
    )
    print("Fetched the URL. The endpoint's behaviour has changed since this was written.")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc)[:300])

HTTP 422, and the message says the fetch itself failed. The endpoint does not reach
out to the public internet for your images.

The fix is the one thing you can always do: download the file in your own code and
encode it, exactly as Step 2 did. If you are passing user supplied URLs, fetching
them yourself is better practice anyway, because it lets you enforce a size limit and
a timeout rather than handing an arbitrary URL to someone else's fetcher.

## Step 4: Reasoning mode

`qwen3-8b` can work through a problem before answering. It is off by default, and you
turn it on with a parameter that is not part of the OpenAI spec, so it travels in
`extra_body`:

```python
extra_body={"chat_template_kwargs": {"enable_thinking": True}}
```

This cell runs the same question three ways. Watch the token counts, not just the
answers.

In [ ]:
QUESTION = (
    "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
    "What does the ball cost? Answer with just the amount."
)

def run(label, thinking, max_tokens):
    kwargs = {}
    if thinking:
        kwargs["extra_body"] = {"chat_template_kwargs": {"enable_thinking": True}}
    r = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": QUESTION}],
        max_tokens=max_tokens,
        temperature=0.0,
        **kwargs,
    )
    msg = r.choices[0].message
    reasoning = getattr(msg, "reasoning", None) or ""
    print(f"{label}")
    print(f"   finish_reason   : {r.choices[0].finish_reason}")
    print(f"   output tokens   : {r.usage.completion_tokens}")
    print(f"   reasoning chars : {len(reasoning)}")
    print(f"   content         : {(msg.content or '')[:70]!r}")
    print()
    return msg

off = run("thinking OFF, max_tokens=250", thinking=False, max_tokens=250)
starved = run("thinking ON,  max_tokens=400", thinking=True, max_tokens=400)
good = run("thinking ON,  max_tokens=2500", thinking=True, max_tokens=2500)

Three things happened, and the middle one is the trap.

**Thinking off.** The answer appears in `content`. The model may still show its
working, but it is doing that in ordinary prose.

**Thinking on with too small a budget.** `content` comes back **empty** and
`finish_reason` is `length`. The reasoning is written first and it consumed the whole
allowance, so the model never reached the answer. You paid for every one of those
tokens and got nothing usable.

**Thinking on with room.** Reasoning lands in `message.reasoning`, the answer lands
in `message.content`, and the two stay separate. You can log the reasoning, show it
behind a toggle, or throw it away, without having to parse it out of the answer.

So the rule is: turning reasoning on means raising `max_tokens` substantially, not
slightly. And check `content` before you use it, because empty content with
`finish_reason: length` is the signature of this exact mistake.

Reasoning is worth it for multi-step arithmetic, logic puzzles, and planning. It is
wasted on lookups and formatting, where it just costs latency and tokens.

## Step 5: Typed output

Asking for JSON in the prompt gets you JSON most of the time. Most of the time is not
good enough to parse. `response_format` with a schema constrains the model so the
output matches your shape.

Build the schema with Pydantic and you get validation for free.

Run the next cell as written. It is supposed to fail, and the way it fails is the most
useful thing in this module.

In [ ]:
from pydantic import BaseModel, Field

class Shape(BaseModel):
    colour: str = Field(description="the colour of the shape")
    kind: str = Field(description="square, circle, or rectangle")

class Shapes(BaseModel):
    total: int = Field(description="how many shapes there are")
    note: str = Field(description="a one line summary")
    shapes: list[Shape]

SCHEMA = {"type": "json_schema", "json_schema": {"name": "shapes", "schema": Shapes.model_json_schema()}}
PROMPT = "A blue square, a red circle, and a yellow rectangle."

naive = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": PROMPT}],
    response_format=SCHEMA,
    max_tokens=300,
    temperature=0.0,
)

content = naive.choices[0].message.content or ""
whitespace = sum(1 for ch in content if ch in " \n\t")

print("finish_reason :", naive.choices[0].finish_reason)
print("output tokens :", naive.usage.completion_tokens)
print(f"content       : {len(content)} chars, {whitespace} of them whitespace")
print(f"\nfirst 120 chars:\n{content[:120]!r}")

`finish_reason: length`, the full 300 token budget spent, and the content is mostly
whitespace. The JSON starts correctly and then dissolves.

Here is what happened. A JSON schema says nothing about whitespace, because JSON
allows any amount of it between tokens. So "emit another space" is always a legal next
move under the grammar. When the model is unsure what should come next, it takes that
legal move, and then takes it again, until the budget runs out. You are billed for
every one of those tokens.

The fix is one sentence in the prompt.

In [ ]:
COMPACT = "Reply with compact JSON on a single line. Do not add whitespace or newlines."

fixed = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": COMPACT},
        {"role": "user", "content": PROMPT},
    ],
    response_format=SCHEMA,
    max_tokens=300,
    temperature=0.0,
)

print("finish_reason :", fixed.choices[0].finish_reason)
print("output tokens :", fixed.usage.completion_tokens, f"(was {naive.usage.completion_tokens})")
print()
print(fixed.choices[0].message.content)

# Parse and type check in one step. Raises if the model got the shape wrong.
result = Shapes.model_validate_json(fixed.choices[0].message.content)
print(f"\nvalidated: {result.total} shapes, note={result.note!r}")
for s in result.shapes:
    print(f"   {s.colour} {s.kind}")

Same schema, same model, same budget. One instruction, and it drops from 300 wasted
tokens to around 30 useful ones.

**Two rules worth keeping.**

Tell the model to emit compact JSON whenever you use `response_format`. It costs you
one line and it removes the failure mode entirely.

Check `finish_reason` before you parse, every time. `length` means the content is
truncated, and truncated JSON is invalid JSON. If you also see that the content is
mostly whitespace, you are looking at the stall above rather than a genuinely long
answer.

`Shapes.model_validate_json` both parses and type checks, so you end up with a real
Python object instead of a dictionary you have to defend against.

One more thing to carry forward. A related stall happens when you combine
`response_format` with tool calling in the same request, and that one cannot be fixed
with a prompt. [Module 04](../04_Agents/) covers it once tools are on the table.

## Validation

In [ ]:
assert (off.content or "").strip(), "thinking-off call produced no content"
assert not (starved.content or "").strip(), "the starved call was expected to return empty content"
assert (getattr(good, "reasoning", None) or ""), "reasoning mode produced no reasoning field"
assert (good.content or "").strip(), "reasoning mode with budget should produce content"
assert fixed.choices[0].finish_reason == "stop", "the compact-JSON call should finish cleanly"
assert result.total == 3 and len(result.shapes) == 3, "structured output did not match the schema"
print("Module 03 complete.")
print("Image input, reasoning mode, and schema-constrained JSON all verified.")
print("You also know the two ways structured output stalls, and how to avoid one of them.")

## Clean up

Nothing was created. No files written, no cloud resources.

## Continue

Next: **[Module 04: Agents](../04_Agents/)**, where the model stops answering
directly and starts asking your code to do things.